In [1]:
#DV: rating row[8]
import numpy as np
#import matplotlib.pyplot as plt
import csv,os
#from sklearn.linear_model import LogisticRegression
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import accuracy_score,precision_score, recall_score
#from sklearn.metrics import confusion_matrix
#from sklearn.inspection import permutation_importance
#import numpy as np
#from sklearn.model_selection import train_test_split
#from sklearn.ensemble import RandomForestClassifier
#from tensorflow.keras.layers import Embedding
from openai import OpenAI

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a batch of texts using OpenAI."""
    response = client.embeddings.create(
        input=texts,
        model=EMBEDDING_MODEL
    )
    return [item.embedding for item in response.data]

#import data from CSVs
data = []
files=["../2_Database/TopBeerData.csv","../2_Database/GoodBeerData.csv","../2_Database/RestBeerData.csv"]
for i in range (0,len(files)):
    with open(files[i],"r",encoding="utf-8") as file:
        reader = csv.reader(file)
        next(reader) # remove header
        for row in reader:
            data.append(row)

all_numeric_data = []
numeric_data = []
for row in data[1:]:  # ignore heading
    numeric_data.append([
        float(row[6]),  # 0 abv
        float(row[9]),   # 1 nr of raters
        float(row[12]),  # 2 malt
        float(row[13]),  # 3 hops
        float(row[14]),  # 4 chocolate
        float(row[15]),  # 5 german
        float(row[16]),  # 6 banana
        float(row[17]),  # 7 vanilla
        float(row[18]),  # 8 caramel
        float(row[19]),  # 9 yeast
        float(row[20]),  # 10 fruit
        float(row[21]),  # 11 whiskey
        float(row[22]),  # 12 rhum
        float(row[23]),  # 13 alcohol
        float(row[24]),  # 14 gluten
        float(row[25]),  # 15 barrel
        float(row[26]),  # 16 rye
        float(row[27]),  # 17 barley
        float(row[28]),  # 18 cocoa
        float(row[29]),  # 19 small/lower
        float(row[30]),  # 20 big/upper
        float(row[31] + row[32] + row[40] + row[49]),  # 21 Altbier and kölsch and Barleywine and Other
        float(row[33] + row[34] + row[35]+ row[36]),  # 22 Blonde and Duppel and Quadrupel and Tripel
        float(row[37] + row[38]),  # 23 Cider and Fruit
        float(row[39]),  # 24 IPA
        float(row[41]),  # 25 Lager
        float(row[42]),  # 26 Non-Alcoholic
        float(row[43]),  # 27 Pilsner
        float(row[44]),  # 28 Porter
        float(row[45]),  # 29 Wheat
        float(row[46]),  # 30 Sour
        float(row[47]),  # 31 Stout
        float(row[48])  # 32 Ale
    ])
numeric_data = np.array(numeric_data)
all_numeric_data.append(numeric_data)
combined = np.vstack(all_numeric_data)
print(combined.shape)

#IDVs + DV def
X = combined[:, :20]
#y = combined[:, 21:33]
y = np.argmax(combined[:, 21:32], axis=1) # ignore 1st and last feature, because those are misleadingly "colorful" groups
#print(X)
print(y)

# -------------------------
# MODEL
# -------------------------

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

EMBEDDING_MODEL = "text-embedding-3-small"
MODEL = "gpt-5-mini"

def embedding_to_bytes(embedding: list[float]) -> bytes:
    """Convert a float list to bytes for SQLite storage."""
    return np.array(embedding, dtype=np.float32).tobytes()

def bytes_to_embedding(blob: bytes) -> np.ndarray:
    """Convert bytes back to a numpy array."""
    return np.frombuffer(blob, dtype=np.float32)

# Build the text to embed for each product
cursor.execute("SELECT id, name, description FROM products")
rows = cursor.fetchall()

texts = [f"{name}: {description}" for _, name, description in rows]
ids = [row[0] for row in rows]

print(f"Generating embeddings for {len(texts)} products...")
embeddings = get_embeddings(texts)
print(f"Each embedding has {len(embeddings[0])} dimensions")

# Store embeddings back in SQLite
for product_id, embedding in zip(ids, embeddings):
    cursor.execute(
        "UPDATE products SET embedding = ? WHERE id = ?",
        (embedding_to_bytes(embedding), product_id)
    )

(2302, 33)
[10  3 10 ...  0  0  0]


NameError: name 'Embedding' is not defined